# Person 1 — Word TF-IDF และ Logistic Regression balanced

โน้ตบุ๊กเฉพาะแนวทางที่ 1 รันได้ด้วยตัวเองบน Google Colab ไม่ต้องเปิดหรือรัน person3
ใช้ CPU ได้ ใน Colab เลือก Runtime > Run all; บนเครื่องทั่วไปใช้ Jupyter และรันจากบนลงล่าง

อ่านข้อมูลจากโครงสร้างเดิม: `egco461-thai-sentiment/code/data/`
บันทึก test เป็น `predictions/dingdong_run1_tfidf_lr_balanced.csv` ภายใต้ project root ที่ค้นพบ
บันทึกคะแนนและ error analysis ที่ `experiments/person1_tfidf_lr_balanced/`

ขั้นตอน: โหลดข้อมูล → สร้างโมเดล → ประเมิน valid/sample และ ablation → error analysis 3 ข้อ → refit train+valid และสร้างไฟล์ส่ง
ใช้ TF-IDF คำ 1–2 grams, Logistic Regression C=2, balanced, seed=461 เหมือนโมเดลเดิม
ไม่ใช้ sample/test ฝึกโมเดล คะแนนประเมินเป็นผลก่อน refit ไม่ใช่คะแนน test


In [1]:
%pip -q install scikit-learn==1.7.2 pythainlp==5.4.0 pandas==2.2.3 numpy==2.2.6 scipy==1.15.3


## 1 ค้นหา project root และโหลดข้อมูล
ใช้วิธีค้นหา root แบบเดียวกับ person3 โดยตรวจว่ามีไฟล์ครบใน `code/data/`
ค้นจากโฟลเดอร์ที่กำลังรัน โฟลเดอร์แม่ และตำแหน่งโปรเจกต์ชื่อ `egco461-thai-sentiment` ภายในระดับที่กำหนด
ใน Colab จะค้นจาก `/content` ด้วย หากยังไม่พบจึงเชื่อมต่อ Drive และค้นจาก MyDrive

บนเครื่องทั่วไป เปิด Jupyter จากโฟลเดอร์โปรเจกต์หรือ `code/` ได้ โดยไม่ต้องติดตั้ง Google Drive
คงโครงสร้าง `code/data/` พร้อม train.csv, valid.csv, sample_labeled.csv และ test.csv
ไฟล์ทำนายจะอยู่ใน `predictions/` ของ root ที่ค้นพบ ซึ่งจะแสดง path เต็มเมื่อโหลดข้อมูลสำเร็จ


In [2]:
import os, warnings
from functools import lru_cache

from pathlib import Path

PROJECT_FOLDER_NAME = 'egco461-thai-sentiment'
REQUIRED_DATA_FILES = ('train.csv', 'valid.csv', 'sample_labeled.csv', 'test.csv')

def _is_project_root(path):
    data_path = path / 'code' / 'data'
    return all((data_path / filename).is_file() for filename in REQUIRED_DATA_FILES)

def _find_project_root(search_base):
    search_base = Path(search_base)
    if not search_base.exists():
        return None
    candidates = [search_base, *search_base.parents]
    for depth in range(4):
        candidates.extend(search_base.glob('*/' * depth + PROJECT_FOLDER_NAME))
    checked = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate not in checked and _is_project_root(candidate):
            return candidate
        checked.add(candidate)
    return None

PROJECT_ROOT = _find_project_root(Path.cwd()) or _find_project_root(Path('/content'))

if PROJECT_ROOT is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_ROOT = _find_project_root(Path('/content/drive/MyDrive'))
    except ImportError:
        pass

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project folder. Extract the project ZIP with its "
        "'code/data/' structure preserved, then run this notebook from that folder "
        "or store the folder in Google Drive."
    )

print('Project:', PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / 'code' / 'data'
PREDICTIONS_DIR = PROJECT_ROOT / 'predictions'
OUTPUT = PROJECT_ROOT / 'experiments' / 'person1_tfidf_lr_balanced'
OUTPUT.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['PYTHAINLP_DATA'] = str(Path('pythainlp_data').resolve())

import numpy as np
import pandas as pd
from IPython.display import display
from pythainlp.tokenize import word_tokenize
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

LABELS = ['pos', 'neu', 'neg', 'q']
def load_csv(filename):
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f'ไม่พบข้อมูล: {path}')
    return pd.read_csv(path, dtype=str, keep_default_na=False)

train = load_csv('train.csv')
valid = load_csv('valid.csv')
sample = load_csv('sample_labeled.csv')
test = load_csv('test.csv')
template_path = DATA_DIR / 'sample_submission.csv'
submission_template = load_csv('sample_submission.csv') if template_path.is_file() else None

for name, data, expected in [('train', train, 24020), ('valid', valid, 2669),
                              ('sample', sample, 100), ('test', test, 1577)]:
    assert len(data) == expected and data.id.is_unique, name
    assert list(data.columns) == (['id', 'text'] if name == 'test' else ['id', 'text', 'category']), name
    if name != 'test': assert set(data.category) == set(LABELS), name
    print(name, len(data))
if submission_template is not None:
    assert test.id.tolist() == submission_template.id.tolist()
print('Data:', DATA_DIR)
print('Test prediction folder:', PREDICTIONS_DIR.resolve())


Mounted at /content/drive
Project: /content/drive/MyDrive/Colab Notebooks/egco461-thai-sentiment
train 24020
valid 2669
sample 100
test 1577
Data: /content/drive/MyDrive/Colab Notebooks/egco461-thai-sentiment/code/data
Test prediction folder: /content/drive/MyDrive/Colab Notebooks/egco461-thai-sentiment/predictions


## 2 สร้างโมเดล
`Pipeline` รวม TF-IDF กับ classifier ไว้ด้วยกัน จึงเรียก fit/predict ได้ตรงไปตรงมา
`fit` เรียน vocabulary และ IDF จาก train เท่านั้น ส่วน valid/sample ใช้ predict เท่านั้น
ไม่ลบคำปฏิเสธ คำซ้ำ หรืออีโมจิ ใช้ lowercase ตามค่าเริ่มต้นของ vectorizer เหมือนโค้ดเดิม
`lru_cache` เก็บผลตัดคำเพื่อประหยัดเวลา โดยไม่ใช้ category และไม่เรียนรู้จากชุดประเมิน


In [3]:
@lru_cache(maxsize=100000)
def tokenize_thai(text):
    return word_tokenize(str(text), engine='newmm', keep_whitespace=False)

def build_model(class_weight):
    vectorizer = TfidfVectorizer(
        tokenizer=tokenize_thai, token_pattern=None,
        ngram_range=(1, 2), min_df=2, max_features=80000,
        sublinear_tf=True, dtype=np.float32,
    )
    classifier = LogisticRegression(
        C=2, class_weight=class_weight,
        solver='lbfgs', max_iter=1500, random_state=461,
    )
    return Pipeline([('tfidf', vectorizer), ('classifier', classifier)])

def fit_checked(model, texts, labels):
    with threadpool_limits(limits=2), warnings.catch_warnings(record=True) as messages:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(texts, labels)
    assert not any(issubclass(m.category, ConvergenceWarning) for m in messages), 'เพิ่ม max_iter แล้วรันใหม่'
    return model

model = fit_checked(build_model('balanced'), train.text, train.category)
model_no_weight = fit_checked(build_model(None), train.text, train.category)


## 3 วัดผลและเปรียบเทียบการถ่วงน้ำหนักคลาส


In [4]:
results = []
predictions = {}
for model_name, fitted_model in [('tfidf_lr', model_no_weight), ('tfidf_lr_balanced', model)]:
    for split_name, data in [('valid', valid), ('sample', sample)]:
        predicted = fitted_model.predict(data.text)
        predictions[(model_name, split_name)] = predicted
        class_f1 = f1_score(data.category, predicted, labels=LABELS, average=None, zero_division=0)
        results.append({
            'model': model_name, 'split': split_name,
            'macro_f1': class_f1.mean(), 'accuracy': accuracy_score(data.category, predicted),
            **dict(zip(['f1_' + label for label in LABELS], class_f1)),
        })
results = pd.DataFrame(results)
display(results.round(6))
results.to_csv(OUTPUT / 'metrics.csv', index=False)
print(classification_report(valid.category, predictions[('tfidf_lr_balanced', 'valid')],
                            labels=LABELS, digits=4, zero_division=0))


,model,split,macro_f1,accuracy,f1_pos,f1_neu,f1_neg,f1_q
0,tfidf_lr,valid,0.555746,0.722743,0.457447,0.787214,0.743029,0.235294
1,tfidf_lr,sample,0.603563,0.700000,0.722222,0.750000,0.608696,0.333333
2,tfidf_lr_balanced,valid,0.605210,0.690521,0.541872,0.733059,0.748318,0.397590
3,tfidf_lr_balanced,sample,0.805276,0.770000,0.700000,0.804124,0.716981,1.000000


              precision    recall  f1-score   support

         pos     0.5102    0.5777    0.5419       476
         neu     0.8058    0.6724    0.7331      1456
         neg     0.6915    0.8152    0.7483       682
           q     0.2973    0.6000    0.3976        55

    accuracy                         0.6905      2669
   macro avg     0.5762    0.6663    0.6052      2669
weighted avg     0.7134    0.6905    0.6959      2669



## 4 Error analysis จริง 3 ตัวอย่าง
เลือกจาก sample_labeled ซึ่งมีเฉลย ห้ามนำ test มาทำ error analysis
เหตุผลด้านล่างเป็นข้อวิเคราะห์จากลักษณะข้อความ ไม่ใช่ข้อพิสูจน์ว่าคำใดเป็นต้นเหตุ
ไม่มีการแก้ label หรือเขียนกฎจาก 3 ตัวอย่างนี้เพื่อปรับผล


In [5]:
analysis_notes = {
    'sp003': ('ประโยคบอกเล่าที่มีคำปฏิเสธ',
              'ยังไม่ได้กินข้าวเลย เป็นการบอกสถานะ ไม่ได้แสดงความไม่พอใจชัดเจน '
              'โมเดลอาจให้น้ำหนักคำ ยัง/ไม่ได้ เป็นสัญญาณลบ; ควรฝึกด้วยประโยคปฏิเสธที่เป็นกลางเพิ่ม'),
    'sp017': ('การประชด',
              'เยี่ยมเลย ขัดกับเหตุการณ์ฝนตกและลืมร่ม จึงสื่อความไม่พอใจ '
              'TF-IDF คำ 1–2 grams แทนความสัมพันธ์ระดับประโยคได้จำกัด; ควรเพิ่มข้อมูลประชดหรือทดลอง contextual model'),
    'sp077': ('สแลงและการสะกดแปรผัน',
              'ดีย์ เป็นการเขียนคำว่า ดี แบบไม่มาตรฐาน และ 555 แสดงเสียงหัวเราะ '
              'คำที่พบน้อยอาจไม่มีใน vocabulary หรือมีน้ำหนักไม่พอ; ควรทดลอง normalization หรือ character features'),
}
errors = sample.assign(prediction=predictions[('tfidf_lr_balanced', 'sample')])
errors = errors[errors.category != errors.prediction].copy()
errors.to_csv(OUTPUT / 'all_sample_errors.csv', index=False, encoding='utf-8-sig')
selected_errors = errors.set_index('id').loc[list(analysis_notes)].reset_index()
selected_errors['cause_group'] = selected_errors.id.map(lambda key: analysis_notes[key][0])
selected_errors['analysis'] = selected_errors.id.map(lambda key: analysis_notes[key][1])
selected_errors.to_csv(OUTPUT / 'error_analysis_3.csv', index=False, encoding='utf-8-sig')
display(selected_errors)


,id,text,category,prediction,cause_group,analysis
0,sp003,ยังไม่ได้กินข้าวเลย,neu,neg,ประโยคบอกเล่าที่มีคำปฏิเสธ,ยังไม่ได้กินข้าวเลย เป็นการบอกสถานะ ไม่ได้แสดง...
1,sp017,เยี่ยมเลย ฝนตกวันที่ลืมพกร่มพอดี,neg,neu,การประชด,เยี่ยมเลย ขัดกับเหตุการณ์ฝนตกและลืมร่ม จึงสื่อ...
2,sp077,โคตรดีย์ 555,pos,neu,สแลงและการสะกดแปรผัน,ดีย์ เป็นการเขียนคำว่า ดี แบบไม่มาตรฐาน และ 55...


## 5 สร้าง prediction ของ test ลงโฟลเดอร์โปรเจกต์
ฝึกโมเดลสุดท้ายใหม่จาก train+valid ด้วย config balanced เดิม ไม่รวม sample/test
ไฟล์ชื่อ `dingdong_run1_tfidf_lr_balanced.csv` อยู่ใน `PROJECT_ROOT/predictions/`
เซลล์นี้อ่านไฟล์กลับและตรวจจำนวนแถว พร้อมแสดง path เต็มเมื่อบันทึกสำเร็จ


In [6]:
final_train = pd.concat([train, valid], ignore_index=True)
print('กำลังฝึกโมเดลสุดท้ายจาก train+valid...', flush=True)
final_model = fit_checked(clone(model), final_train.text, final_train.category)
submission = pd.DataFrame({'id': test.id, 'category': final_model.predict(test.text)})

assert list(submission.columns) == ['id', 'category']
assert len(submission) == 1577 and submission.id.is_unique
assert submission.id.tolist() == test.id.tolist()
if submission_template is not None:
    assert submission.id.tolist() == submission_template.id.tolist()
assert submission.category.isin(LABELS).all() and not submission.isna().any().any()

prediction_path = PREDICTIONS_DIR / 'dingdong_run1_tfidf_lr_balanced.csv'
submission.to_csv(prediction_path, index=False, encoding='utf-8')
saved = pd.read_csv(prediction_path, dtype=str, keep_default_na=False)
assert saved.equals(submission) and len(saved) == 1577

import json, hashlib, platform
import sklearn, pythainlp, scipy
config = {
    'model': 'word TF-IDF + LogisticRegression balanced',
    'seed': 461, 'C': 2, 'class_weight': 'balanced',
    'solver': 'lbfgs', 'max_iter': 1500,
    'ngram_range': [1, 2], 'min_df': 2, 'max_features': 80000,
    'sublinear_tf': True, 'final_training_rows': len(final_train),
    'final_training_data': 'train + valid; exclude sample_labeled/test',
    'python': platform.python_version(), 'sklearn': sklearn.__version__,
    'pythainlp': pythainlp.__version__, 'pandas': pd.__version__,
    'numpy': np.__version__, 'scipy': scipy.__version__,
    'prediction_sha256': hashlib.sha256(prediction_path.read_bytes()).hexdigest(),
}
(OUTPUT / 'run_config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print('บันทึกและอ่านกลับสำเร็จ:', prediction_path.resolve())
print('จำนวนข้อมูล:', len(saved), 'แถว | ขนาดไฟล์:', prediction_path.stat().st_size, 'bytes')
print('ผลทดลองและ error analysis:', OUTPUT)
print('CSV ในโฟลเดอร์ predictions:')
for path in sorted(PREDICTIONS_DIR.glob('*.csv')):
    print(' -', path.name)


กำลังฝึกโมเดลสุดท้ายจาก train+valid...
บันทึกและอ่านกลับสำเร็จ: /content/drive/MyDrive/Colab Notebooks/egco461-thai-sentiment/predictions/dingdong_run1_tfidf_lr_balanced.csv
จำนวนข้อมูล: 1577 แถว | ขนาดไฟล์: 17231 bytes
ผลทดลองและ error analysis: /content/drive/MyDrive/Colab Notebooks/egco461-thai-sentiment/experiments/person1_tfidf_lr_balanced
CSV ในโฟลเดอร์ predictions:
 - dingdong_run1_tfidf_lr_balanced.csv
 - dingdong_run3_pos_neu-logit_bias_wangchanberta.csv


## ไฟล์ผลลัพธ์
- `predictions/dingdong_run1_tfidf_lr_balanced.csv`: คำทำนาย test สำหรับส่ง
- `experiments/person1_tfidf_lr_balanced/metrics.csv`: ผล valid/sample และ ablation
- `experiments/person1_tfidf_lr_balanced/error_analysis_3.csv`: ตัวอย่างผิด 3 ข้อพร้อมคำวิเคราะห์
- `experiments/person1_tfidf_lr_balanced/all_sample_errors.csv`: errors ทั้งหมดบน sample
- `experiments/person1_tfidf_lr_balanced/run_config.json`: ค่าตั้งต้นและเวอร์ชันที่รัน

เปิดโฟลเดอร์ตาม path ที่พิมพ์จากเซลล์สุดท้ายเพื่อดูไฟล์ หาก root อยู่ใน Drive ผลลัพธ์จะบันทึกลง Drive
ผล sample มีเพียง 100 ข้อความ ไม่ใช่คะแนน test
